# nl2cuda-kernel-skill — Colab 一站式验证 notebook

配合 `USAGE.md` **第一部分·路径 A** 使用。运行时选 **T4 GPU**（`代码执行程序 → 更改运行时类型 → T4 GPU`）。

**只有两个 cell，从上到下跑**：
1. **冒烟 cell**：首次弹窗上传 `skill_delivery.tgz`（本地打的 skill 包，见 USAGE A-2.5）+ 装 ninja + 跑内置 rbf（verify/bench），确认环境就绪。
2. **你的 case cell**：把本地 agent 产的 case 打包成 base64（见 USAGE A-2），填进 `CASE`/`B64` 两个空，一个 cell 跑完解包 + verify + bench。

> ⚠️ Colab(外网)连不到内网仓,所以 skill 走上传、不 clone。闲置约 90 分钟/断网会**重置运行时**(仓库/case/缓存全丢)——重跑 Cell 1 会再次弹窗上传 skill 包即可恢复。
> T4 上 rbf 前向可能 bench FAIL 属正常(前向优势要 A100 才显现),T4 验的是**正确性 + 链路通**。

## Cell 1 · 冒烟：确认环境（跑内置 rbf 样例）
首次 nvcc 编译 rbf 要等几分钟、中途无输出正常。rbf 的 verify 全 PASS 即环境就绪。

In [ ]:
import os, tarfile
os.chdir('/content')
REPO = '/content/nl2cuda-kernel-skill'
PKG  = '/content/skill_delivery.tgz'
# Colab(外网)连不到内网仓,skill 包走上传:本地 `tar czf skill_delivery.tgz -C <交付目录> .` 生成后,
# 跑本 cell 会自动弹窗让你选该文件(或提前用左侧文件面板传到 /content 也行)。
if not os.path.isdir(REPO):
    if not os.path.exists(PKG):
        from google.colab import files
        print('请在弹窗中选择本地的 skill_delivery.tgz ...')
        up = files.upload()   # 弹窗:选 skill_delivery.tgz
        # 取上传到的文件名(通常就是 skill_delivery.tgz;兼容重名如 (1))
        cand = [f for f in up if f.endswith('.tgz')] or list(up)
        if not cand:
            raise SystemExit('未上传任何文件。请重跑本 cell 并选择 skill_delivery.tgz')
        PKG = '/content/' + cand[0]
    os.makedirs(REPO, exist_ok=True)
    with tarfile.open(PKG) as t: t.extractall(REPO)
    print('已解包 skill 包:', PKG)
os.chdir(REPO)
!pip install ninja -q
!python scripts/probe_env.py                      # 确认 GPU / CUDA / PyTorch
!python framework/smoke_test.py                   # 确认 nvcc + ninja 编译链路
!python skill/scripts/verify_case.py --case rbf   # 冒烟：前反向 5 种子应全 PASS
!python skill/scripts/bench_case.py  --case rbf   # 冒烟：跑通计时基准即可

## Cell 2 · 你的 case：解包 + verify + bench（一次性）

先在**本地**让 agent 产出 `cases/<你的算法名>/`，按 USAGE **A-2** 打包成一行 base64。然后：
- 把 `CASE` 改成你的 case 名（如 `rmsnorm`，**不要留尖括号**）；
- 把 `B64` 粘上那一整行 base64（结尾通常是 `==`）；
- 跑本 cell。改一版就重打包、重跑本 cell（幂等，仓库在就跳过 clone）。

> 若 `bench` 报「短核假象警告」（baseline <1ms），在下面补一个 cell 放大规模复测：
> `!<规模ENV>=262144 python skill/scripts/bench_case.py --case 你的case名`（`<规模ENV>` 见该 case 的 `config.py`，如 RMSNorm 是 `RMS_B`）。

In [ ]:
import os, tarfile
os.chdir('/content')
REPO = '/content/nl2cuda-kernel-skill'
PKG  = '/content/skill_delivery.tgz'
# Colab(外网)连不到内网仓,skill 包走上传:本地 `tar czf skill_delivery.tgz -C <交付目录> .` 生成后,
# 跑本 cell 会自动弹窗让你选该文件(或提前用左侧文件面板传到 /content 也行)。
if not os.path.isdir(REPO):
    if not os.path.exists(PKG):
        from google.colab import files
        print('请在弹窗中选择本地的 skill_delivery.tgz ...')
        up = files.upload()   # 弹窗:选 skill_delivery.tgz
        # 取上传到的文件名(通常就是 skill_delivery.tgz;兼容重名如 (1))
        cand = [f for f in up if f.endswith('.tgz')] or list(up)
        if not cand:
            raise SystemExit('未上传任何文件。请重跑本 cell 并选择 skill_delivery.tgz')
        PKG = '/content/' + cand[0]
    os.makedirs(REPO, exist_ok=True)
    with tarfile.open(PKG) as t: t.extractall(REPO)
    print('已解包 skill 包:', PKG)
os.chdir(REPO)
!pip install ninja -q

import base64, io, shutil
CASE = '你的算法名'                        # ← 改成实际 case 名,如 rmsnorm(不要留尖括号)
B64  = '在此粘贴本地 agent 产的 case 那一整行 base64'  # ← 一整行,结尾通常是 ==

# 清旧 case 目录 + torch 编译缓存(改了 kernel 重跑时避免用到旧版)
shutil.rmtree(os.path.join(REPO, 'cases', CASE), ignore_errors=True)
shutil.rmtree('/root/.cache/torch_extensions', ignore_errors=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64))) as t:
    t.extractall('.')
print('case 文件:', os.listdir(f'cases/{CASE}'))
!python skill/scripts/verify_case.py --case {CASE}
!python skill/scripts/bench_case.py  --case {CASE}